In [1]:
import preprocess
import run_cicero

In [2]:
import scanpy as sc
import pandas as pd
from scipy.io import mmread

In [3]:
counts = mmread("../make_cicero_cds_R_Objs/full_matrix.mtx").T.tocsr()
obs_names = pd.read_csv("../make_cicero_cds_R_Objs/full_meta.csv", header = 0, index_col = 0)
var_names = pd.read_csv("../make_cicero_cds_R_Objs/full_matrix_rownames.csv", header = 0, index_col = 1)
var_names.drop("Unnamed: 0", axis = 1, inplace = True)
# obs_names.drop("Unnamed: 0", axis = 1, inplace = True)

In [4]:
adata = sc.AnnData(counts, obs = obs_names, var = var_names)

In [5]:
adata

AnnData object with n_obs × n_vars = 37584 × 266716
    obs: 'orig.ident', 'nCount_ATAC', 'nFeature_ATAC', 'nCount_RNA', 'nFeature_RNA', 'segment', 'nucleosome_signal', 'nucleosome_percentile', 'TSS.enrichment', 'TSS.percentile', 'nCount_SCT', 'nFeature_SCT', 'SCT.weight', 'ATAC.weight', 'cell_type', 'cluster', 'group'

In [5]:
adata = preprocess.preprocess_cicero(adata)

preprocess.py: 2025-04-16 03:15:14 WARNING  Counts layers not found. Coppying X to counts


preprocess.py: 2025-04-16 03:15:15 INFO     Estimaing Size Factors and Normalizing Data
preprocess.py: 2025-04-16 03:15:21 INFO     Finished Size Factors and Normallization storing normalized sparse matrix into 'data'
preprocess.py: 2025-04-16 03:15:21 INFO     Running TF-IDF
preprocess.py: 2025-04-16 03:15:25 INFO     Finsihed TF-IDF
preprocess.py: 2025-04-16 03:15:25 INFO     Running TruncatedSVD
preprocess.py: 2025-04-16 03:17:54 INFO     Finsihed TruncatedSVD
preprocess.py: 2025-04-16 03:17:54 INFO     Running UMAP
preprocess.py: 2025-04-16 03:17:56 INFO     Finsihed UMAP


In [6]:
cicero_adata = preprocess.make_cicero_adata(adata, k = 50)

preprocess.py: 2025-04-16 03:17:56 INFO     Using adata OBSM X_umap to agregate cells
preprocess.py: 2025-04-16 03:17:56 INFO     Calculating overlap
preprocess.py: 2025-04-16 03:17:56 INFO     Generating Pseudobulk cicero observations with seed: 0 and k: 50
preprocess.py: 2025-04-16 03:18:12 INFO     Reached Maximum itterations in pseudobulk observation generation. Consider increasing 'max_itterations'
preprocess.py: 2025-04-16 03:18:12 INFO     Found 4484 good choices
preprocess.py: 2025-04-16 03:18:12 INFO     Finished calculating overlap
preprocess.py: 2025-04-16 03:18:12 INFO     Aggregating Cells
preprocess.py: 2025-04-16 03:18:28 INFO     Finished Aggregating Cells


In [7]:
# Preprocess the index strings, assuming format "chr-start-end" (e.g., "chr-0-1923") 
# Also sorted from lower to higher
temp_df = pd.DataFrame([x.split("-") for x in cicero_adata.var.index])
cicero_adata.var["Chromosome"] = temp_df.iloc[:, 0].values
cicero_adata.var["Start"] = temp_df.iloc[:, 1].astype(int).values
cicero_adata.var["End"] = temp_df.iloc[:, 2].astype(int).values
cicero_adata.var["Mean"] = (cicero_adata.var["Start"] + cicero_adata.var["End"])/2

In [8]:
#Usually sorted, if not sort
chromosomes_ordered = ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8',
       'chr9', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15',
       'chr16', 'chr17', 'chr18', 'chr19', 'chrX', 'chrY', 'GL456211.1',
       'GL456216.1', 'GL456233.1', 'JH584295.1', 'JH584304.1'] #doesn't actually matter
temp_vars = []
for chromosome in chromosomes_ordered:
    chromosome_var = cicero_adata.var[cicero_adata.var["Chromosome"] == chromosome]
    chromosome_var.sort_values("Start", ascending = False) #lower to higher
    temp_vars.append(chromosome_var)
cicero_adata.var = pd.concat(temp_vars, axis = 0)

In [9]:
cicero_adata.var

,Chromosome,Start,End,Mean
x,,,,
chr1-3119740-3120239,chr1,3119740,3120239,3119989.5
chr1-3121226-3121725,chr1,3121226,3121725,3121475.5
chr1-3155081-3155580,chr1,3155081,3155580,3155330.5
chr1-3203852-3204351,chr1,3203852,3204351,3204101.5
chr1-3210121-3210620,chr1,3210121,3210620,3210370.5
...,...,...,...,...
JH584304.1-67266-67765,JH584304.1,67266,67765,67515.5
JH584304.1-67886-68385,JH584304.1,67886,68385,68135.5
JH584304.1-68747-69246,JH584304.1,68747,69246,68996.5


In [11]:
cicero_adata

AnnData object with n_obs × n_vars = 4484 × 266716
    obs: 'orig.ident', 'nCount_ATAC', 'nFeature_ATAC', 'nCount_RNA', 'nFeature_RNA', 'segment', 'nucleosome_signal', 'nucleosome_percentile', 'TSS.enrichment', 'TSS.percentile', 'nCount_SCT', 'nFeature_SCT', 'SCT.weight', 'ATAC.weight', 'cell_type', 'cluster', 'group', 'aggregate_obs_names'
    var: 'Chromosome', 'Start', 'End', 'Mean'

In [10]:
"""
probably from inverse_covriance (skggm)
from: Starting distance_parameter_estimation
/home/twoo/miniconda3/envs/rapids_SC/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
"""
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, message=".*force_all_finite.*")

In [65]:
import importlib
importlib.reload(run_cicero)

<module 'run_cicero' from '/home/twoo/work/spinal_cord/epigenomic_spinal_cord/pyCicero/pyCicero/run_cicero.py'>

In [73]:
cons = run_cicero.run_cicero(cicero_adata)

run_cicero.py: 2025-04-15 17:49:39 INFO     Generating Windows
run_cicero.py: 2025-04-15 17:49:39 INFO     Starting distance_parameter_estimation
100%|██████████| 500/500 [05:29<00:00,  1.52it/s]
run_cicero.py: 2025-04-15 17:55:30 INFO     Starting multiprocessing pool for estimate_distance_parameter_parallel with 96 processes
run_cicero.py: 2025-04-15 18:04:47 INFO     Finished distance_parameter_estimation
run_cicero.py: 2025-04-15 18:04:47 INFO     Starting generate_cicero_models
run_cicero.py: 2025-04-15 18:05:44 INFO     Starting Cicero with 96 processes
run_cicero.py: 2025-04-15 18:05:56 WARNING  diag(V) had non-positive or NA entries; the non-finite result may be dubious
run_cicero.py: 2025-04-15 18:07:00 WARNING  diag(V) had non-positive or NA entries; the non-finite result may be dubious
run_cicero.py: 2025-04-15 18:07:10 INFO     Finished generate_cicero_models
run_cicero.py: 2025-04-15 18:07:12 INFO     Starting assemble_connections
run_cicero.py: 2025-04-15 18:07:35 INFO   

In [74]:
cons[cons["coaccess_score"] != 0]

,Peak1,Peak2,coaccess_score
0,GL456211.1-112603-113102,GL456211.1-199065-199564,-0.789865
1,GL456216.1-16175-16674,GL456216.1-16819-17318,0.999968
12,GL456216.1-31947-32446,GL456216.1-32548-33047,0.988294
19,GL456216.1-33752-34251,GL456216.1-34703-35202,0.982985
22,GL456233.1-105020-105519,GL456233.1-109964-110463,0.044250
...,...,...,...
9164623,chrY-90804920-90805419,chrY-90808559-90809058,0.742748
9164624,chrY-90804920-90805419,chrY-90810437-90810936,0.310894
9164626,chrY-90807503-90808002,chrY-90808559-90809058,0.814334
9164627,chrY-90807503-90808002,chrY-90810437-90810936,0.340858


In [37]:
import pickle
pickle.dump(cons, open("cons.pickle", "wb"))